## 准备数据

In [2]:
import os
import numpy as np
import tensorflow as tf
from tensorflow import keras
from keras import layers, optimizers, datasets

os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'  # or any {'0', '1', '2'}

def mnist_dataset():
    (x, y), (x_test, y_test) = datasets.mnist.load_data()
    #normalize
    x = x/255.0
    x_test = x_test/255.0
    
    return (x, y), (x_test, y_test)

In [3]:
print(list(zip([1, 2, 3, 4], ['a', 'b', 'c', 'd'])))

[(1, 'a'), (2, 'b'), (3, 'c'), (4, 'd')]


## 建立模型

In [4]:
class myModel:
    def __init__(self):
        ####################
        '''声明模型对应的参数'''
        ####################
        self.W1 = tf.Variable(tf.random.normal((784, 512)))
        self.b1 = tf.Variable(tf.zeros((512,)))
        self.W2 = tf.Variable(tf.random.normal((512, 128)))
        self.b2 = tf.Variable(tf.zeros((128,)))
        self.W3 = tf.Variable(tf.random.normal((128, 10)))
        self.b3 = tf.Variable(tf.zeros((10,)))
    def __call__(self, x):
        ####################
        '''实现模型函数体，返回未归一化的logits'''
        ####################
        x = tf.reshape(x, (-1, 784))  # 展平输入
        x = tf.nn.relu(tf.matmul(x, self.W1) + self.b1)  # 第一个全连接层
        x = tf.nn.relu(tf.matmul(x, self.W2) + self.b2)  # 第二个全连接层
        logits = tf.matmul(x, self.W3) + self.b3  # 第三个全连接层
        return logits
        
model = myModel()

optimizer = optimizers.Adam()

## 计算 loss

In [5]:
@tf.function
def compute_loss(logits, labels):
    return tf.reduce_mean(
        tf.nn.sparse_softmax_cross_entropy_with_logits(
            logits=logits, labels=labels))

@tf.function
def compute_accuracy(logits, labels):
    predictions = tf.argmax(logits, axis=1)
    return tf.reduce_mean(tf.cast(tf.equal(predictions, labels), tf.float32))

@tf.function
def train_one_step(model, optimizer, x, y):
    with tf.GradientTape() as tape:
        logits = model(x)
        loss = compute_loss(logits, y)

    # compute gradient
    trainable_vars = [model.W1, model.W2, model.b1, model.b2]
    grads = tape.gradient(loss, trainable_vars)
    for g, v in zip(grads, trainable_vars):
        v.assign_sub(0.01*g)

    accuracy = compute_accuracy(logits, y)

    # loss and accuracy is scalar tensor
    return loss, accuracy

@tf.function
def test(model, x, y):
    logits = model(x)
    loss = compute_loss(logits, y)
    accuracy = compute_accuracy(logits, y)
    return loss, accuracy

## 实际训练

In [9]:
train_data, test_data = mnist_dataset()
for epoch in range(50):
    loss, accuracy = train_one_step(model, optimizer, 
                                    tf.constant(train_data[0], dtype=tf.float32), 
                                    tf.constant(train_data[1], dtype=tf.int64))
    print('epoch', epoch, ': loss', loss.numpy(), '; accuracy', accuracy.numpy())
loss, accuracy = test(model, 
                      tf.constant(test_data[0], dtype=tf.float32), 
                      tf.constant(test_data[1], dtype=tf.int64))

print('test loss', loss.numpy(), '; accuracy', accuracy.numpy())

epoch 0 : loss 43.9096 ; accuracy 0.84435
epoch 1 : loss 43.74779 ; accuracy 0.8448333
epoch 2 : loss 43.58785 ; accuracy 0.8454
epoch 3 : loss 43.4298 ; accuracy 0.84583336
epoch 4 : loss 43.273666 ; accuracy 0.84608334
epoch 5 : loss 43.1192 ; accuracy 0.84635
epoch 6 : loss 42.966404 ; accuracy 0.8466667
epoch 7 : loss 42.814907 ; accuracy 0.8472
epoch 8 : loss 42.664837 ; accuracy 0.8476167
epoch 9 : loss 42.516422 ; accuracy 0.84816664
epoch 10 : loss 42.36958 ; accuracy 0.84845
epoch 11 : loss 42.223892 ; accuracy 0.84855
epoch 12 : loss 42.07929 ; accuracy 0.84891665
epoch 13 : loss 41.936115 ; accuracy 0.8494667
epoch 14 : loss 41.794308 ; accuracy 0.8498667
epoch 15 : loss 41.653767 ; accuracy 0.8502167
epoch 16 : loss 41.51464 ; accuracy 0.85066664
epoch 17 : loss 41.37715 ; accuracy 0.851
epoch 18 : loss 41.2409 ; accuracy 0.8513333
epoch 19 : loss 41.106133 ; accuracy 0.85165
epoch 20 : loss 40.97274 ; accuracy 0.8519
epoch 21 : loss 40.840847 ; accuracy 0.85223335
epoch 22